# 02 — Casablanca A-E Zone Remapping (v4)

**Pipeline:**
1. Load v4 arrondissement polygons (9 OSM + 7 Voronoi, clipped to city boundary)
2. Join with per-arrondissement HCP 2024 population
3. Spatial-join Glovo H3-res9 commerce cells → per-zone commerce metrics
4. Count tram/BRT stops within each zone
5. Compute **A-E urban activity score** = `0.30·commerce + 0.40·pop_density + 0.20·poi_diversity + 0.10·transit_hubs`
6. Quantile-cut into A/B/C/D/E classes (~2/4/5/4/1 of 16)
7. Persist `data/zone_mapping_v4.csv` and render `notebooks/casablanca_ae_map.html`

In [1]:
from __future__ import annotations
import json, sys, math
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import shape, Point
import folium
from folium.plugins import Fullscreen

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'scripts'))
from hcp_loader import load_province_priors, load_arrondissement_population

print('ROOT =', ROOT)

ROOT = c:\Users\hp\OneDrive - Université Abdelmalek Essaadi\Desktop\Taasimm


## 1. Load all ground-truth data

In [2]:
# Arrondissement polygons (16 zones)
zones_gdf = gpd.read_file(ROOT / 'data' / 'casablanca_arrondissements_v4.geojson').set_crs(4326)
print(f'Zones: {len(zones_gdf)}')
zones_gdf[['zone_id', 'name', 'osm_id', 'source']].head(16)

Zones: 16


,zone_id,name,osm_id,source
0,1,Ain Chock,2801442.0,osm_relation
1,2,Sidi Othmane,NaN,voronoi_clipped_city
2,3,Sidi Moumen,NaN,voronoi_clipped_city
3,4,Hay Hassani,4743065.0,osm_relation
4,5,Sbata,2801415.0,osm_relation
5,6,Ben Msik,2801410.0,osm_relation
6,7,Moulay Rachid,NaN,voronoi_clipped_city
7,8,Maarif,2801474.0,osm_relation
8,9,Al Fida,2801452.0,osm_relation
9,10,Mers Sultan,NaN,voronoi_clipped_city


In [3]:
# HCP per-arrondissement population
pop_df = load_arrondissement_population()
print(f'Population CSV: {len(pop_df)} zones, total = {pop_df.population_2024.sum():,}')
pop_df.head()

Population CSV: 16 zones, total = 3,189,000


,zone_id,arrondissement_name,population_2024,prefecture,note
0,1,Ain Chock,374000,Ain Chock,HCP RGPH 2024 portal (screenshot)
1,2,Sidi Othmane,211000,Ben Msick / Moulay Rachid,HCP RGPH 2024 portal (screenshot)
2,3,Sidi Moumen,551000,Sidi Bernoussi,HCP RGPH 2024 portal (screenshot) - combined w...
3,4,Hay Hassani,536000,Hay Hassani,HCP RGPH 2024 portal (screenshot)
4,5,Sbata,101000,Ben Msick,HCP RGPH 2024 portal (screenshot)


In [4]:
# HCP province-level priors (applied uniformly as calibration constants)
priors = load_province_priors()
print(f'HCP priors: {len(priors)} indicators')
for k, v in list(priors.items())[:6]:
    print(f'  {k:<30} {v}')

HCP priors: 18 indicators
  population_province            3199416.0
  population_female              1628412.0
  population_male                1571004.0
  share_under_15_pct             21.2
  share_15_59_pct                61.9
  share_60_plus_pct              16.9


In [5]:
# Glovo commerce H3 cells
commerce_gdf = gpd.read_file(ROOT / 'data' / 'opportunity-find' / 'glovo_opportunity_score_from_commerce-2.geojson').set_crs(4326)
print(f'Glovo H3 cells: {len(commerce_gdf)}')
commerce_gdf.head(3)

Glovo H3 cells: 934


,h3_index,opportunity_score,commerce_count,commerce_weight_sum,commerce_types,geometry
0,8939aab9417ffff,3.6,27,14.6,shop|pharmacy|restaurant|office|hotel|cafe|ban...,"POLYGON ((-7.62082 33.59544, -7.62243 33.5941,..."
1,8939aab94abffff,9.9,94,57.7,nightclub|pharmacy|craft|shop|restaurant|bar|f...,"POLYGON ((-7.61708 33.59627, -7.61868 33.59493..."
2,8939aa1654bffff,0.3,1,0.5,shop,"POLYGON ((-7.64379 33.53695, -7.64539 33.5356,..."


In [6]:
# Transit geometry
transit_gdf = gpd.read_file(ROOT / 'data' / 'casablanca_transit.geojson').set_crs(4326)
print(transit_gdf.groupby('mode').size())

mode
brt           16
brt_stop       3
tram         216
tram_stop    218
dtype: int64


## 2. Compute per-zone area (km²) and join population

In [7]:
# Project to metric CRS (Morocco Lambert) for area measurement
zones_metric = zones_gdf.to_crs(3857)
zones_gdf['area_km2'] = (zones_metric.area / 1e6).round(3)
zones_gdf = zones_gdf.merge(pop_df[['zone_id', 'population_2024']], on='zone_id', how='left')
zones_gdf['pop_density'] = (zones_gdf['population_2024'] / zones_gdf['area_km2']).round(1)
zones_gdf[['zone_id','name','area_km2','population_2024','pop_density']]

,zone_id,name,area_km2,population_2024,pop_density
0,1,Ain Chock,56.881,374000,6575.1
1,2,Sidi Othmane,7.503,211000,28122.1
2,3,Sidi Moumen,24.400,551000,22582.0
3,4,Hay Hassani,58.740,536000,9125.0
4,5,Sbata,10.337,101000,9770.7
5,6,Ben Msik,4.488,105000,23395.7
6,7,Moulay Rachid,12.223,244000,19962.4
7,8,Maarif,17.903,139000,7764.1
8,9,Al Fida,5.481,125000,22806.1
9,10,Mers Sultan,4.640,97000,20905.2


## 3. Spatial-join Glovo commerce H3 cells to zones

In [8]:
# Use the H3-cell centroid as the spatial key to avoid duplicate zone hits from overlap.
commerce_pts = commerce_gdf.copy()
commerce_pts['geometry'] = commerce_pts.geometry.centroid
joined = gpd.sjoin(commerce_pts, zones_gdf[['zone_id', 'name', 'geometry']], how='inner', predicate='within')
print(f'H3 cells inside Casa zones: {len(joined)} / {len(commerce_gdf)}')

def _poi_diversity(series: pd.Series) -> int:
    types = set()
    for cell in series.dropna():
        types.update(str(cell).split('|'))
    types.discard('')
    return len(types)

commerce_by_zone = joined.groupby('zone_id').agg(
    commerce_count_sum=('commerce_count', 'sum'),
    commerce_weight_sum=('commerce_weight_sum', 'sum'),
    opportunity_score_mean=('opportunity_score', 'mean'),
    h3_cell_count=('h3_index', 'count'),
    poi_diversity=('commerce_types', _poi_diversity),
).reset_index()
commerce_by_zone

H3 cells inside Casa zones: 903 / 934


C:\Users\hp\AppData\Local\Temp\ipykernel_23368\2349934256.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  commerce_pts['geometry'] = commerce_pts.geometry.centroid


,zone_id,commerce_count_sum,commerce_weight_sum,opportunity_score_mean,h3_cell_count,poi_diversity
0,1,423,164.3,0.581752,137,20
1,2,81,28.8,0.536667,30,14
2,3,67,24.8,0.416667,36,13
3,4,595,246.1,0.664634,164,21
4,5,78,23.7,0.600000,22,10
5,6,63,17.2,0.552381,21,11
6,7,83,27.6,0.463158,38,13
7,8,868,385.6,1.358163,98,20
8,9,94,31.7,0.669231,26,12
9,10,142,58.2,0.996000,25,15


## 4. Count transit hubs within each zone

In [9]:
stops = transit_gdf[transit_gdf['mode'].isin(['tram_stop', 'brt_stop'])].copy()
stops = gpd.sjoin(stops, zones_gdf[['zone_id', 'geometry']], how='inner', predicate='within')
transit_by_zone = stops.groupby('zone_id').size().rename('transit_hub_count').reset_index()
transit_by_zone

,zone_id,transit_hub_count
0,1,8
1,2,14
2,3,13
3,4,26
4,5,5
5,6,6
6,7,18
7,8,22
8,9,14
9,10,7


## 5. Assemble per-zone feature matrix

In [10]:
z = zones_gdf[['zone_id','name','area_km2','population_2024','pop_density']].copy()
z = z.merge(commerce_by_zone, on='zone_id', how='left').fillna({'commerce_count_sum': 0, 'commerce_weight_sum': 0, 'opportunity_score_mean': 0, 'h3_cell_count': 0, 'poi_diversity': 0})
z = z.merge(transit_by_zone, on='zone_id', how='left').fillna({'transit_hub_count': 0})
z[['transit_hub_count','commerce_count_sum','poi_diversity']] = z[['transit_hub_count','commerce_count_sum','poi_diversity']].astype(int)
z

,zone_id,name,area_km2,population_2024,pop_density,commerce_count_sum,commerce_weight_sum,opportunity_score_mean,h3_cell_count,poi_diversity,transit_hub_count
0,1,Ain Chock,56.881,374000,6575.1,423,164.3,0.581752,137,20,8
1,2,Sidi Othmane,7.503,211000,28122.1,81,28.8,0.536667,30,14,14
2,3,Sidi Moumen,24.400,551000,22582.0,67,24.8,0.416667,36,13,13
3,4,Hay Hassani,58.740,536000,9125.0,595,246.1,0.664634,164,21,26
4,5,Sbata,10.337,101000,9770.7,78,23.7,0.600000,22,10,5
5,6,Ben Msik,4.488,105000,23395.7,63,17.2,0.552381,21,11,6
6,7,Moulay Rachid,12.223,244000,19962.4,83,27.6,0.463158,38,13,18
7,8,Maarif,17.903,139000,7764.1,868,385.6,1.358163,98,20,22
8,9,Al Fida,5.481,125000,22806.1,94,31.7,0.669231,26,12,14
9,10,Mers Sultan,4.640,97000,20905.2,142,58.2,0.996000,25,15,7


## 6. Normalize metrics and compute A-E activity score

`score = 0.30·commerce + 0.40·pop_density + 0.20·poi_diversity + 0.10·transit_hubs`

In [11]:
def _minmax(s: pd.Series) -> pd.Series:
    lo, hi = s.min(), s.max()
    return (s - lo) / (hi - lo) if hi > lo else pd.Series([0.5]*len(s), index=s.index)

z['commerce_n']   = _minmax(z['commerce_weight_sum'])
z['pop_density_n']= _minmax(z['pop_density'])
z['poi_div_n']    = _minmax(z['poi_diversity'])
z['transit_n']    = _minmax(z['transit_hub_count'])

WEIGHTS = {'commerce_n': 0.30, 'pop_density_n': 0.40, 'poi_div_n': 0.20, 'transit_n': 0.10}
z['ae_score'] = sum(z[c] * w for c, w in WEIGHTS.items()).round(4)
z[['zone_id','name','ae_score','commerce_n','pop_density_n','poi_div_n','transit_n']].sort_values('ae_score', ascending=False)

,zone_id,name,ae_score,commerce_n,pop_density_n,poi_div_n,transit_n
13,14,Sidi Belyout,0.6728,1.000000,0.193390,1.0000,0.954545
1,2,Sidi Othmane,0.5007,0.017431,1.000000,0.2500,0.454545
7,8,Maarif,0.4257,0.553569,0.132047,0.6250,0.818182
3,4,Hay Hassani,0.4167,0.343952,0.190069,0.6875,1.000000
2,3,Sidi Moumen,0.3874,0.011420,0.763801,0.1875,0.409091
8,9,Al Fida,0.3863,0.021788,0.773355,0.1250,0.454545
9,10,Mers Sultan,0.3715,0.061608,0.692311,0.3125,0.136364
6,7,Moulay Rachid,0.3667,0.015627,0.652116,0.1875,0.636364
5,6,Ben Msik,0.3410,0.000000,0.798492,0.0625,0.090909
10,11,Roches Noires,0.2751,0.133434,0.218425,0.3750,0.727273


In [12]:
# Quantile cuts: A=top 10%, B=25%, C=30%, D=25%, E=bottom 10% -> on 16 zones -> 2/4/5/4/1
ranked = z.sort_values('ae_score', ascending=False).reset_index(drop=True)
bucket_counts = [2, 4, 5, 4, 1]  # A, B, C, D, E
assert sum(bucket_counts) == 16
labels = ['A']*2 + ['B']*4 + ['C']*5 + ['D']*4 + ['E']*1
ranked['ae_class'] = labels
z = z.merge(ranked[['zone_id','ae_class']], on='zone_id')
z.groupby('ae_class').size()

ae_class
A    2
B    4
C    5
D    4
E    1
dtype: int64

## 7. Persist `data/zone_mapping_v4.csv`

In [13]:
# Compute centroids from v4 polygons (not legacy) for consistency
zc = zones_gdf.copy()
zc['centroid_lat'] = zc.geometry.centroid.y.round(5)
zc['centroid_lon'] = zc.geometry.centroid.x.round(5)
out = z.merge(zc[['zone_id','centroid_lat','centroid_lon']], on='zone_id')

# Carry adjacency from legacy zone_mapping.csv
legacy = pd.read_csv(ROOT / 'data' / 'zone_mapping.csv')[['zone_id','adjacent_zones']]
out = out.merge(legacy, on='zone_id', how='left')

cols = ['zone_id','name','centroid_lat','centroid_lon','area_km2','population_2024','pop_density',
        'commerce_count_sum','commerce_weight_sum','opportunity_score_mean','h3_cell_count',
        'poi_diversity','transit_hub_count','ae_score','ae_class','adjacent_zones']
out = out[cols].sort_values('zone_id').reset_index(drop=True)
out.to_csv(ROOT / 'data' / 'zone_mapping_v4.csv', index=False)
print(f'Wrote {ROOT / "data" / "zone_mapping_v4.csv"}')
out

Wrote c:\Users\hp\OneDrive - Université Abdelmalek Essaadi\Desktop\Taasimm\data\zone_mapping_v4.csv


C:\Users\hp\AppData\Local\Temp\ipykernel_23368\1185286145.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  zc['centroid_lat'] = zc.geometry.centroid.y.round(5)
C:\Users\hp\AppData\Local\Temp\ipykernel_23368\1185286145.py:4: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  zc['centroid_lon'] = zc.geometry.centroid.x.round(5)


,zone_id,name,centroid_lat,centroid_lon,area_km2,population_2024,pop_density,commerce_count_sum,commerce_weight_sum,opportunity_score_mean,h3_cell_count,poi_diversity,transit_hub_count,ae_score,ae_class,adjacent_zones
0,1,Ain Chock,33.52657,-7.62155,56.881,374000,6575.1,423,164.3,0.581752,137,20,8,0.2420,C,"2,4,5,6,8,9"
1,2,Sidi Othmane,33.55833,-7.56133,7.503,211000,28122.1,81,28.8,0.536667,30,14,14,0.5007,A,"3,5,6,7,9,10,11,12"
2,3,Sidi Moumen,33.58382,-7.50041,24.400,551000,22582.0,67,24.8,0.416667,36,13,13,0.3874,B,"2,7,15,16"
3,4,Hay Hassani,33.54646,-7.68025,58.740,536000,9125.0,595,246.1,0.664634,164,21,26,0.4167,B,"1,8,13"
4,5,Sbata,33.53576,-7.55799,10.337,101000,9770.7,78,23.7,0.600000,22,10,5,0.0945,E,"1,2,6"
5,6,Ben Msik,33.55447,-7.58130,4.488,105000,23395.7,63,17.2,0.552381,21,11,6,0.3410,C,"1,2,5,9"
6,7,Moulay Rachid,33.56927,-7.53666,12.223,244000,19962.4,83,27.6,0.463158,38,13,18,0.3667,C,"2,3,12,15"
7,8,Maarif,33.57043,-7.63251,17.903,139000,7764.1,868,385.6,1.358163,98,20,22,0.4257,B,"1,4,9,10,13,14"
8,9,Al Fida,33.56519,-7.59490,5.481,125000,22806.1,94,31.7,0.669231,26,12,14,0.3863,B,"1,6,8,10,11"
9,10,Mers Sultan,33.57672,-7.60385,4.640,97000,20905.2,142,58.2,0.996000,25,15,7,0.3715,C,"2,8,9,11,14"


## 8. Folium choropleth: A-E classes + H3 overlay + tram lines

In [15]:
AE_COLORS = {'A': '#b30000', 'B': '#e34a33', 'C': '#fc8d59', 'D': '#fdcc8a', 'E': '#fef0d9'}
center = [33.575, -7.6]
m = folium.Map(location=center, zoom_start=12, tiles='cartodbpositron', control_scale=True)
Fullscreen().add_to(m)

# zones_gdf already has population_2024 + pop_density; only bring in the NEW columns
zones_for_map = zones_gdf.merge(
    out[['zone_id','ae_class','ae_score','commerce_count_sum','transit_hub_count']],
    on='zone_id'
)

def _style(feat):
    c = feat['properties']['ae_class']
    return {'fillColor': AE_COLORS.get(c, '#ccc'), 'fillOpacity': 0.65, 'color': '#333', 'weight': 1.2}

folium.GeoJson(
    zones_for_map.to_json(),
    name='Arrondissements (A-E)',
    style_function=_style,
    tooltip=folium.GeoJsonTooltip(
        fields=['zone_id','name','ae_class','ae_score','population_2024','pop_density','commerce_count_sum','transit_hub_count'],
        aliases=['#','Arrondissement','Class','Score','Population','Pop/km²','Commerce','Transit hubs']
    ),
).add_to(m)

# Tram lines
tram_lines = transit_gdf[transit_gdf['mode'] == 'tram']
folium.GeoJson(
    tram_lines.to_json(),
    name='Tram lines',
    style_function=lambda _f: {'color': '#1f77b4', 'weight': 3, 'opacity': 0.9},
).add_to(m)

# Tram stops (small dots)
tstops_layer = folium.FeatureGroup(name='Tram stops')
for _, row in transit_gdf[transit_gdf['mode'] == 'tram_stop'].iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=2.2, color='#1f77b4', fill=True, fill_opacity=0.9,
        tooltip=row.get('name') or 'tram stop',
    ).add_to(tstops_layer)
tstops_layer.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
out_html = ROOT / 'notebooks' / 'casablanca_ae_map.html'
m.save(str(out_html))
print(f'Saved map -> {out_html}')
m

Saved map -> c:\Users\hp\OneDrive - Université Abdelmalek Essaadi\Desktop\Taasimm\notebooks\casablanca_ae_map.html


## 9. Validation

In [16]:
# Sanity checks
assert out['ae_class'].value_counts().to_dict() == {'A': 2, 'B': 4, 'C': 5, 'D': 4, 'E': 1}
assert out['population_2024'].sum() == pop_df['population_2024'].sum()
assert (out['area_km2'] > 0).all()
print('All validations passed.')
print('\nTop 5 A-class candidates:')
print(out.sort_values('ae_score', ascending=False)[['name','ae_class','ae_score','population_2024','commerce_count_sum','transit_hub_count']].head(5).to_string(index=False))

All validations passed.

Top 5 A-class candidates:
        name ae_class  ae_score  population_2024  commerce_count_sum  transit_hub_count
Sidi Belyout        A    0.6728           136000                1201                 25
Sidi Othmane        A    0.5007           211000                  81                 14
      Maarif        B    0.4257           139000                 868                 22
 Hay Hassani        B    0.4167           536000                 595                 26
 Sidi Moumen        B    0.3874           551000                  67                 13
